# Notebook 1 — Data Inspection (LendingClub Accepted Loans 2007–2018Q4)

**Project:** Credit Risk & KPI Decision System  
**Goal (this notebook):** Load the raw CSV and produce a clean inspection report: schema, missingness, target distribution, and basic sanity checks.

---

## Contents
1. Imports & settings  
2. File paths & quick  
3. Load strategy 
4. Schema & summary statistics  
5. Duplicates & identifiers  
6. Missingness report  
7. Outcome field checks (`loan_status`)  
8. Date coverage checks  
9. Common string-encoded numerics  
10. Data quality checklist  
11. Save inspection artefacts  

In [1]:
# Imports & settings
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [2]:
DATA_PATH = Path("data/raw/accepted_2007_to_2018Q4.csv")
assert DATA_PATH.exists(), f"File not found: {DATA_PATH.resolve()}"
DATA_PATH

PosixPath('data/raw/accepted_2007_to_2018Q4.csv')

In [3]:
# Preview a few rows to inspect column names and formats
sample = pd.read_csv(DATA_PATH, nrows=5, low_memory=False)
sample.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,leadman,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,debt_consolidation,Debt consolidation,190xx,PA,5.91,0.0,Aug-2003,675.0,679.0,1.0,30.0,NaN,7.0,0.0,2765.0,29.7,13.0,w,0.00,0.00,4421.723917,4421.72,3600.00,821.72,0.0,0.0,0.0,Jan-2019,122.67,NaN,Mar-2019,564.0,560.0,0.0,30.0,1.0,Individual,NaN,NaN,NaN,0.0,722.0,144904.0,2.0,2.0,0.0,1.0,21.0,4981.0,36.0,3.0,3.0,722.0,34.0,9300.0,3.0,1.0,4.0,4.0,20701.0,1506.0,37.2,0.0,0.0,148.0,128.0,3.0,3.0,1.0,4.0,69.0,4.0,69.0,2.0,2.0,4.0,2.0,5.0,3.0,4.0,9.0,4.0,7.0,0.0,0.0,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,Engineer,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,n,https://lendingclub.com/browse/loanDetail.acti...,NaN,small_business,Business,577xx,SD,16.06,1.0,Dec-1999,715.0,719.0,4.0,6.0,NaN,22.0,0.0,21470.0,19.2,38.0,w,0.00,0.00,25679.660000,25679.66,24700.00,979.66,0.0,0.0,0.0,Jun-2016,926.35,NaN,Mar-2019,699.0,695.0,0.0,NaN,1.0,Individual,NaN,NaN,NaN,0.0,0.0,204396.0,1.0,1.0,0.0,1.0,19.0,18005.0,73.0,2.0,3.0,6472.0,29.0,111800.0,0.0,0.0,6.0,4.0,9733.0,57830.0,27.1,0.0,0.0,113.0,192.0,2.0,2.0,4.0,2.0,NaN,0.0,6.0,0.0,5.0,5.0,13.0,17.0,6.0,20.0,27.0,5.0,22.0,0.0,0.0,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0,NaN,NaN,NaN,NaN,NaN,N

In [5]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.shape

(2260701, 151)

## Schema & summary statistics
Inspect datatypes, non-null counts, and broad statistical ranges. This helps spot obvious parsing issues.

In [ ]:
df.info(verbose=True, show_counts=True)

## Duplicates & identifiers
Check duplicate rows and common ID-like fields.


In [8]:
dup_pct = df.duplicated().mean() * 100
dup_pct

np.float64(0.0)

In [9]:
id_like = [c for c in df.columns if c.lower() in {"id", "member_id"}]
id_like

['id', 'member_id']

In [10]:
for c in id_like:
    print(c, "unique:", df[c].nunique(dropna=True), "| nulls:", int(df[c].isna().sum()))

id unique: 2260701 | nulls: 0
member_id unique: 0 | nulls: 2260701


## Outcome field (`loan_status`)
Later notebooks will define the exact mapping to **default** vs **non-default**.

In [14]:
"loan_status" in df.columns

True

In [17]:
df["loan_status"].value_counts(dropna=False)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [19]:
(df["loan_status"].value_counts(normalize=True, dropna=False) * 100).round(2)

loan_status
Fully Paid                                             47.63
Current                                                38.85
Charged Off                                            11.88
Late (31-120 days)                                      0.95
In Grace Period                                         0.37
Late (16-30 days)                                       0.19
Does not meet the credit policy. Status:Fully Paid      0.09
Does not meet the credit policy. Status:Charged Off     0.03
Default                                                 0.00
NaN                                                     0.00
Name: proportion, dtype: float64

## Date coverage checks
LendingClub often stores dates as strings. parse key date columns safely and check min/max coverage.

In [20]:
date_cols = [c for c in df.columns if c.endswith("_d")]
date_cols

['issue_d', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']

In [24]:
DATE_FORMAT = "%b-%Y"

for c in ["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"]:
    if c in df.columns:
        df[c + "_parsed"] = pd.to_datetime(df[c], format=DATE_FORMAT, errors="coerce")
        print(c, df[c + "_parsed"].min(), "->", df[c + "_parsed"].max())

issue_d 2007-06-01 00:00:00 -> 2018-12-01 00:00:00
earliest_cr_line 1933-03-01 00:00:00 -> 2015-11-01 00:00:00
last_pymnt_d 2007-12-01 00:00:00 -> 2019-03-01 00:00:00
last_credit_pull_d 2007-05-01 00:00:00 -> 2019-04-01 00:00:00


In [25]:
df["issue_year"] = df["issue_d_parsed"].dt.year
df["issue_month"] = df["issue_d_parsed"].dt.month

df["earliest_cr_year"] = df["earliest_cr_line_parsed"].dt.year

## Common string-encoded numeric fields
These frequently appear as strings (percent signs, text, etc.). We just preview values here.
Cleaning + casting will happen in the cleaning/SQL staging notebook.


In [23]:
suspects = ["int_rate", "revol_util", "term", "emp_length"]
for c in suspects:
    if c in df.columns:
        print("\n", c)
        print(df[c].astype(str).head(10).tolist())


 int_rate
['13.99', '11.99', '10.78', '14.85', '22.45', '13.44', '9.17', '8.49', '6.49', '11.48']

 revol_util
['29.7', '19.2', '56.2', '11.6', '64.5', '68.4', '84.5', '5.7', '34.5', '39.1']

 term
[' 36 months', ' 36 months', ' 60 months', ' 60 months', ' 60 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months']

 emp_length
['10+ years', '10+ years', '10+ years', '10+ years', '3 years', '4 years', '10+ years', '10+ years', '6 years', '10+ years']


## Data quality checklist

In [26]:
quality_checks = []

def add_check(name, value):
    quality_checks.append({"check": name, "result": value})

add_check("rows", df.shape[0])
add_check("columns", df.shape[1])
add_check("duplicate_row_pct", round(df.duplicated().mean() * 100, 3))

if "loan_status" in df.columns:
    add_check("loan_status_unique", df["loan_status"].nunique(dropna=True))
    add_check("loan_status_nulls", int(df["loan_status"].isna().sum()))

if "issue_d_parsed" in df.columns:
    add_check("issue_d_min", str(df["issue_d_parsed"].min()))
    add_check("issue_d_max", str(df["issue_d_parsed"].max()))

pd.DataFrame(quality_checks)

,check,result
0,rows,2260701
1,columns,158
2,duplicate_row_pct,0.0
3,loan_status_unique,9
4,loan_status_nulls,33
5,issue_d_min,2007-06-01 00:00:00
6,issue_d_max,2018-12-01 00:00:00


## Save inspection artefacts

**Saved files:**
- `missingness.csv`
- `columns.csv`
- `loan_status_counts.csv`


In [27]:
OUTPUT_DIR = Path("data/inspection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing_table.to_csv(OUTPUT_DIR / "missingness.csv")
pd.Series(df.columns, name="columns").to_csv(OUTPUT_DIR / "columns.csv", index=False)

if "loan_status" in df.columns:
    df["loan_status"].value_counts(dropna=False).to_csv(OUTPUT_DIR / "loan_status_counts.csv")

OUTPUT_DIR

PosixPath('data/inspection')